In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os, sys

# Add repo root (one level above notebooks/) to sys.path
nb_dir = Path(os.getcwd()).resolve()
repo_root = nb_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

out_dir = Path(repo_root) / "output" / "BEM_Brazilian_disk"
out_dir.mkdir(parents=True, exist_ok=True)

from fracture_utils.Ubem.bem_solver import BEMSolver2D
from fracture_utils.Ubem.boundary_conditions import (
    build_boundary,
    BCSpec,
    assemble_segment_bcs,
    add_boundary_to_solver,
)
from fracture_utils.Ubem.bem_plotter import plot_centerline_stresses_circle
from fracture_utils.Ubem.bem_stress_field import circle_inside
from fracture_utils.Ubem.bem_stress_plotter import (
    ContourOpts,
    eval_and_plot_stress_components_contours_separate,
)
from fracture_utils.Ubem.bem_brazilian_disk_analytic import (
    stress_brazilian_disk,
    stress_center,
    l2_error_on_line
)

# -------------------------
# Inputs
# -------------------------
R = 25.4e-3/2
P_total = 3.8e3
E = 231.52e9
nu = 0.3
n_elem = 90
arc_half_angle_deg = 15.0
h = 6.35e-3
P = 2*P_total

# -------------------------
# 1) Build + discretize boundary
# -------------------------
boundary_spec = {"type": "circle", "R": R, "n_boundary": n_elem, "center": (0.0, 0.0)}
mesh = build_boundary(boundary_spec)

# -------------------------
# 2) Define BCs (distributed normal pressure on two arcs)
# -------------------------
theta_deg = np.asarray(mesh.theta_deg, dtype=float)
L = np.asarray(mesh.length, dtype=float)

top = (theta_deg >= 90 - arc_half_angle_deg) & (theta_deg <= 90 + arc_half_angle_deg)
bot = (theta_deg >= -90 - arc_half_angle_deg) & (theta_deg <= -90 + arc_half_angle_deg)

L_top = float(np.sum(L[top]))
L_bot = float(np.sum(L[bot]))

if L_top <= 0 or L_bot <= 0:
    raise RuntimeError("Top/bottom loaded arc has zero length — check selector ranges or n_elem.")

pressure_top = P_total / L_top
pressure_bot = P_total / L_bot

bc_specs = [
    BCSpec(
        load_type="pressure_normal",
        bc_value=pressure_top,
        selector_kind="theta_deg_range",
        selector_data=(90 - arc_half_angle_deg, 90 + arc_half_angle_deg),
    ),
    BCSpec(
        load_type="pressure_normal",
        bc_value=pressure_bot,
        selector_kind="theta_deg_range",
        selector_data=(-90 - arc_half_angle_deg, -90 + arc_half_angle_deg),
    ),
]
is_traction, bc_x, bc_y = assemble_segment_bcs(
    mesh,
    bc_specs=bc_specs,
    default=("traction", 0.0, 0.0),
)

# -------------------------
# 3) Solve
# -------------------------
solver = BEMSolver2D(E=E, nu=nu, h=h, plane_strain=True)
add_boundary_to_solver(solver, mesh, is_traction, bc_x, bc_y)
solver.solve(gauss_n=4)

# -------------------------
# 4) Plot stresses along centerline (numerical)
# -------------------------
plot_path = plot_centerline_stresses_circle(
    solver,
    R=R,
    out_dir=out_dir,
    basename="brazilian_disk_line_stresses",
    normalize_by=P_total,  # your plot convention: stress/P_total
    n_points=100,
)
print("Saved:", plot_path)

# -------------------------
# 5) Contour stress maps (numerical)
# -------------------------
bbox = (-R, R, -R, R)
inside = circle_inside(R, center=(0.0, 0.0), pad=0.03*R)

opts_common = dict(
    dpi=300,
    show=True,
    robust=True,
    robust_pct=97.0,
    n_levels=30,
    n_line_levels=20,
    x_scale=1e3, y_scale=1e3,      # m -> mm
    x_label="x [mm]", y_label="y [mm]",
    value_scale=1e-6,              # Pa -> MPa
    cbar_label="Stress [MPa]",
    cmap="jet",                 # recommended diverging stress colormap
)

opts_xx = ContourOpts(**opts_common, title="σ_xx", symmetric=True)
opts_yy = ContourOpts(**opts_common, title="σ_yy", symmetric=True)
opts_xy = ContourOpts(**opts_common, title="σ_xy", symmetric=True)

xs, ys, Sxx, Syy, Sxy, paths = eval_and_plot_stress_components_contours_separate(
    solver,
    bbox=bbox,
    out_dir=out_dir,
    basename="brazilian_disk",
    n=120,
    inside=inside,
    normalize_by=None,   # keep in Pa; value_scale converts to MPa for display
    opts_xx=opts_xx,
    opts_yy=opts_yy,
    opts_xy=opts_xy,
)

np.save(out_dir / "Sxx.npy", Sxx)
np.save(out_dir / "Syy.npy", Syy)
np.save(out_dir / "Sxy.npy", Sxy)
np.save(out_dir / "xs.npy", xs)
np.save(out_dir / "ys.npy", ys)

print("Saved contour plots:", paths)
print("Saved arrays: Sxx.npy, Syy.npy, Sxy.npy, xs.npy, ys.npy")



Saved: C:\Users\Owner\Documents\Repos\Fracture\VCM_3_1\output\BEM_Brazilian_disk\brazilian_disk_line_stresses.png


In [ ]:
# ============================================================
# UPDATED Cell 2 (v2): Uncoupled BEM -> spatial applied stress
# Uses the STRESS GRID saved by Cell 1 (xs/ys/Sxx/Syy/Sxy) and
# interpolates stresses at crack quadrature/collocation points.
# Notes:
# - This is still "uncoupled" physics (no boundary correction).
# - For maximum accuracy (and for iterative/direct coupling),
#   we will later replace this interpolator with an operator-based
#   evaluator built directly from the BEM boundary solution.
# ============================================================

import os, sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

nb_dir = Path(os.getcwd()).resolve()
repo_root = nb_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from preamble import *

# ---- USE YOUR MODULE (no scratch code)
from fracture_utils.Ugenerator.crack_network_simplifier import CrackNetworkSimplifier, SimplificationConfig
from fracture_utils.Uplotter.core import DCEPlotterV4

out_dir = Path(repo_root) / "output" / "KinkedCrackPropagation"
out_dir.mkdir(parents=True, exist_ok=True)


import inspect
import fracture_utils.Uplotter.core as core_mod

print("core_mod file:", core_mod.__file__)
print("DCEPlotterV4 imported from:", inspect.getfile(DCEPlotterV4))
print("Has plot_stress_components_global on class?",
      hasattr(DCEPlotterV4, "plot_stress_components_global"))

# also check the instance
_tmp = DCEPlotterV4(res, out_dir=out_dir) if "res" in globals() else None
if _tmp is not None:
    print("Has plot_stress_components_global on instance?",
          hasattr(_tmp, "plot_stress_components_global"))
    print("plot-related methods:",
          [n for n in dir(_tmp) if "stress" in n.lower() or "plot" in n.lower()])



# ============================================================
# 0) Coupling mode (this cell implements "uncoupled")
# ============================================================
BEM_coupling_mode = "uncoupled"   # {"uncoupled","iterative","direct"} later
bem_field_dir = Path(repo_root) / "output" / "BEM_Brazilian_disk"

# ============================================================
# 0b) Build a pointwise stress evaluator from saved BEM grids
# ============================================================
def _build_bem_sigma_func_from_saved_grid(bem_dir: Path):
    xs  = np.load(bem_dir / "xs.npy")
    ys  = np.load(bem_dir / "ys.npy")
    Sxx = np.load(bem_dir / "Sxx.npy")
    Syy = np.load(bem_dir / "Syy.npy")
    Sxy = np.load(bem_dir / "Sxy.npy")

    # Replace NaNs (outside disk) with 0 for interpolation robustness.
    # (Crack points should be inside disk in this demo.)
    Sxx0 = np.nan_to_num(Sxx, nan=0.0)
    Syy0 = np.nan_to_num(Syy, nan=0.0)
    Sxy0 = np.nan_to_num(Sxy, nan=0.0)

    # Prefer SciPy interpolators if available
    try:
        from scipy.interpolate import RegularGridInterpolator

        # Grid is (ys, xs) in array storage with meshgrid(indexing="xy")
        Ixx = RegularGridInterpolator((ys, xs), Sxx0, bounds_error=False, fill_value=0.0)
        Iyy = RegularGridInterpolator((ys, xs), Syy0, bounds_error=False, fill_value=0.0)
        Ixy = RegularGridInterpolator((ys, xs), Sxy0, bounds_error=False, fill_value=0.0)

        def sigma_components(X):
            X = np.asarray(X, float)
            # RegularGridInterpolator expects points as (y, x) tuples when axes are (ys,xs)
            pts = np.column_stack([X[:, 1], X[:, 0]])
            sxx = Ixx(pts)
            syy = Iyy(pts)
            sxy = Ixy(pts)
            return sxx, syy, sxy

        return sigma_components

    except Exception:
        # SciPy not available; fall back to bilinear interpolation manually.
        # This assumes xs and ys are sorted and uniform-ish.
        xs = np.asarray(xs, float); ys = np.asarray(ys, float)

        def _interp2(Z, xq, yq):
            # bilinear on rect grid; Z shape (ny,nx) with axes (ys,xs)
            xq = np.asarray(xq, float); yq = np.asarray(yq, float)
            ix = np.searchsorted(xs, xq) - 1
            iy = np.searchsorted(ys, yq) - 1
            ix = np.clip(ix, 0, len(xs)-2)
            iy = np.clip(iy, 0, len(ys)-2)

            x0 = xs[ix]; x1 = xs[ix+1]
            y0 = ys[iy]; y1 = ys[iy+1]
            tx = (xq - x0) / np.maximum(x1 - x0, 1e-300)
            ty = (yq - y0) / np.maximum(y1 - y0, 1e-300)

            z00 = Z[iy,   ix]
            z10 = Z[iy,   ix+1]
            z01 = Z[iy+1, ix]
            z11 = Z[iy+1, ix+1]

            return (1-tx)*(1-ty)*z00 + tx*(1-ty)*z10 + (1-tx)*ty*z01 + tx*ty*z11

        def sigma_components(X):
            X = np.asarray(X, float)
            xq = X[:, 0]; yq = X[:, 1]
            sxx = _interp2(Sxx0, xq, yq)
            syy = _interp2(Syy0, xq, yq)
            sxy = _interp2(Sxy0, xq, yq)
            return sxx, syy, sxy

        return sigma_components

_bem_sigma_components = _build_bem_sigma_func_from_saved_grid(bem_field_dir)

def sigma_tensor_from_bem_grid(X):
    sxx, syy, sxy = _bem_sigma_components(X)
    S = np.zeros((len(X), 2, 2), float)
    S[:, 0, 0] = sxx
    S[:, 1, 1] = syy
    S[:, 0, 1] = sxy
    S[:, 1, 0] = sxy
    return S

# ============================================================
# Build initial network
# ============================================================
mm = 1e-3
vertices = np.array([
    [0,   -2.0*mm,  -1.0*mm],
    [1,    2.0*mm,   1.0*mm],
    [2,    -3.0*mm,   3.0*mm],
    [3,    1.0*mm,   -2.0*mm]
], dtype=float)

connectivity = np.array([[0, 1], [2, 3]], dtype=int)

net0 = CrackNetworkV4.from_vertices_connectivity(
    vertices=vertices,
    connectivity=connectivity,
    Nv_max=4,
    validate=True
)

material = Material(E=200e9, nu=0.30, plane_stress=False)

# Applied stress: spatial (from BEM grid) in uncoupled mode
applied  = AppliedStress(sigma_func=sigma_tensor_from_bem_grid)

# ----------------------------
# ONE solver kwargs dict (used everywhere)
# ----------------------------
solver_kwargs = dict(
    ne_half=80,
    representation="cheb_quad",
    node_distribution="tip_dense",
    solver_option="parametrized_crack",
    parametrization="polyline",
    nq_col=12,
    nq_stress=16,
    crack_mode="half",
    junction_model="core",
    refine_kinks=False,
    refine_junction_endpoints=False,
    endpoint_min_nodes=4,
    kink_side_nodes=1,
    tip_cluster="power",
    tip_cluster_power=2.5,
    other_min_panels=4,
    # IMPORTANT: plotting later should not add a constant remote field
)

# ----------------------------
# Solve + plot (only when called)
# ----------------------------
def solve_and_plot(network, tag: str):
    step_dir = out_dir / tag
    step_dir.mkdir(parents=True, exist_ok=True)

    calc = DCENetworkStaticV4(material, network, applied)
    sol  = calc.solve(**solver_kwargs)
    res  = DCEResultsNetworkV4(calc, sol)

    plotter = DCEPlotterV4(res, out_dir=step_dir)
    _call(plotter, "plot_network_graph")

    opts = StressPlotOptsV4(
        extent_factor=3,
        n_grid=301,
        add_remote=False,  # <-- because applied stress is spatial, not constant
        mask_cracks=False,
        cmap="jet",
        n_bands=40,
        label_contours=False,
        vmax_factor=2,
    )
    _call(plotter, "plot_stress_components_global", opts=opts, components=("sxx", "syy", "sxy"))

    pl_def = DCEPlotterDeformedV4(res, out_dir=step_dir)
    _call(pl_def, "plot_deformed_network", scale=5e1, trim_core_junction_faces=True)

    plt.show()
    print(f"[{tag}] Saved figures to: {step_dir}")
    return res

# ----------------------------
# Propagation setup (unchanged)
# ----------------------------
cfg = PropagationConfig(f0=0.05, f_fixed=0.05,
                        step_mode="fixed_step", simultaneous_tip_growth=True)

Kc_demo = 1e6  # Pa*sqrt(m)
tough = ConstantToughness(Kc_demo)

dir_law = MaximumHoopStressLaw()

evaluator = CandidateEvaluator(
    material=material,
    applied=applied,
    solver_kwargs=solver_kwargs,
    direction_law=dir_law,
    enable_ne_half_escalation=False,
)
evaluator.rmax_frac = 0.25
evaluator.min_pts   = 8

prop = CrackPropagator(
    cfg=cfg,
    evaluator=evaluator,
    toughness=tough,
    direction_law=dir_law,
)

# ============================================================
# Helpers: export network + chain length (single polyline)
# ============================================================
def net_to_arrays(net: CrackNetworkV4):
    V = np.array([[int(v.id), float(v.x), float(v.y)] for v in net.vertices], dtype=float)
    C = np.array([[int(e.v0), int(e.v1)] for e in net.edges], dtype=int)
    return V, C

def chain_order_from_connectivity(C):
    adj = defaultdict(list)
    deg = defaultdict(int)
    for a, b in C:
        a = int(a); b = int(b)
        adj[a].append(b); adj[b].append(a)
        deg[a] += 1; deg[b] += 1
    tips = [vid for vid, d in deg.items() if d == 1]
    start = tips[0] if tips else int(C[0, 0])
    order = []
    prev = None
    cur = start
    while True:
        order.append(cur)
        nbrs = adj[cur]
        nxt = None
        if prev is None:
            nxt = nbrs[0] if nbrs else None
        else:
            cand = [x for x in nbrs if x != prev]
            nxt = cand[0] if cand else None
        if nxt is None:
            break
        prev, cur = cur, nxt
        if len(order) > len(adj) + 5:
            break
    return order

def polyline_length_mm_from_arrays(V, C):
    if C.size == 0:
        return 0.0
    xy = {int(row[0]): np.array([row[1], row[2]], float) for row in V}
    order = chain_order_from_connectivity(C)
    L = 0.0
    for i in range(len(order) - 1):
        L += float(np.linalg.norm(xy[order[i+1]] - xy[order[i]]))
    return L / mm

# ============================================================
# Outer-cycle controls
# ============================================================
vertex_limit = 20
L_limit_mm   = 20.0
max_cycles   = 8

simp_cfg = SimplificationConfig(
    max_angle_deviation=3.0,
    min_edge_length=0.25*mm,
    remove_degree2_nodes=False,
    merge_at_degree2=True,
    merge_vertex_tolerance=1e-6,
    max_merged_edge_length=10*mm,
    preserve_tips=True,
    preserve_junctions=True
)

# ----------------------------
# Plot INITIAL
# ----------------------------
res = solve_and_plot(net0, tag="cycle_00_initial")

# ----------------------------
# Outer-cycle loop
# ----------------------------
net = net0
global_step = 0

for cyc in range(1, max_cycles + 1):
    vertex_high = 18
    vertex_low  = 12

    did_grow_this_cycle = False

    while True:
        V_now, C_now = net_to_arrays(net)
        Lmm = polyline_length_mm_from_arrays(V_now, C_now)
        if Lmm >= L_limit_mm:
            print(f"[cycle_{cyc:02d}] reached length limit: L={Lmm:.2f} mm")
            break

        result = prop.grow_one_increment(net)
        net = result.network_new

        grew = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
        global_step += 1
        did_grow_this_cycle |= (grew > 0)

        V_now, C_now = net_to_arrays(net)
        nV = V_now.shape[0]
        Lmm = polyline_length_mm_from_arrays(V_now, C_now)

        print(f"[cycle_{cyc:02d} step_{global_step:04d}] grown_tips={grew} | nV={nV} | L={Lmm:.2f} mm")

        if grew == 0:
            print(f"[cycle_{cyc:02d}] No tips exceeded toughness. Stopping.")
            break

        if nV >= vertex_high:
            print(f"[cycle_{cyc:02d}] hit high-water vertex threshold: nV={nV} >= {vertex_high}")
            break

    # ---- end-of-cycle outputs (PRE-simplify)
    V_pre, C_pre = net_to_arrays(net)
    Lmm_pre = polyline_length_mm_from_arrays(V_pre, C_pre)

    cyc_dir = out_dir / f"cycle_{cyc:02d}"
    cyc_dir.mkdir(parents=True, exist_ok=True)

    np.savez(cyc_dir / f"network_pre_simplify_step_{global_step:04d}.npz",
             vertices=V_pre, connectivity=C_pre)

    res = solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_pre_simplify")

    if Lmm_pre >= L_limit_mm:
        print(f"[cycle_{cyc:02d}] terminating outer loop (length reached).")
        break

    if 'result' in locals():
        grew_last = sum(1 for r in result.reports if bool(getattr(r, "grew", False)))
        if grew_last == 0:
            print(f"[cycle_{cyc:02d}] terminating outer loop (no growth).")
            break

    # ---- simplify using YOUR MODULE
    simplifier = CrackNetworkSimplifier(config=simp_cfg)
    simplifier.load_from_arrays(V_pre, C_pre)
    simp_stats = simplifier.simplify(verbose=True)
    V_post, C_post = simplifier.to_arrays()

    np.savez(cyc_dir / f"network_post_simplify_step_{global_step:04d}.npz",
             vertices=V_post, connectivity=C_post)

    net = CrackNetworkV4.from_vertices_connectivity(
        vertices=V_post,
        connectivity=C_post,
        Nv_max=4,
        validate=True
    )

    res = solve_and_plot(net, tag=f"cycle_{cyc:02d}_step_{global_step:04d}_post_simplify")

print("Saved figures under:", out_dir)


In [ ]:
# ============================================================
# Cell 3 (style-matched): Combined BEM + Crack stress contours (UNCoupled)
# - Computes crack stresses on the SAME (xs,ys) grid and saves arrays
# - Superposes: TOTAL = BEM + Crack  (Pa)
# - Plots TOTAL using the SAME contour/threshold logic as DCEPlotterV4
#   (robust vmin/vmax, n_bands, contour lines, cmap, etc.)
#
# Assumptions:
#   - Cell 1 saved xs.npy, ys.npy, Sxx.npy, Syy.npy, Sxy.npy under output/BEM_Brazilian_disk/
#   - Cell 2 produced `res` (DCEResultsNetworkV4) in the current kernel
#   - You installed the fixed core.py where plot_stress_components_global is a DCEPlotterV4 method
# ============================================================

import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

nb_dir = Path(os.getcwd()).resolve()
repo_root = nb_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from preamble import *
import fracture_utils.Uplotter.core as core_mod
from fracture_utils.Uplotter.core import DCEPlotterV4, StressPlotOptsV4

bem_dir = repo_root / "output" / "BEM_Brazilian_disk"
out_dir = bem_dir / "combined_uncoupled"
out_dir.mkdir(parents=True, exist_ok=True)

# --- Load BEM grid + stresses (Pa)
xs = np.load(bem_dir / "xs.npy")
ys = np.load(bem_dir / "ys.npy")
Sxx_bem = np.load(bem_dir / "Sxx.npy")
Syy_bem = np.load(bem_dir / "Syy.npy")
Sxy_bem = np.load(bem_dir / "Sxy.npy")

# --- Compute crack stresses on SAME grid, crack-only (no remote)
plotter = DCEPlotterV4(res, out_dir=out_dir)

opts = StressPlotOptsV4(
    extent_factor=3,      # unused when grid override is provided
    n_grid=len(xs),
    add_remote=False,     # IMPORTANT: crack-only; BEM is the applied field
    mask_cracks=False,
    cmap="jet",
    n_bands=40,
    label_contours=False,
    vmax_factor=2,
    dpi=300,
)

plotter.plot_stress_components_global(
    opts=opts,
    components=("sxx", "syy", "sxy"),
    grid=(xs, ys),
    save_arrays=True,
    arrays_prefix="crack",
    show=False,
    save=False,
)

Sxx_cr = np.load(out_dir / "crack_sxx.npy")
Syy_cr = np.load(out_dir / "crack_syy.npy")
Sxy_cr = np.load(out_dir / "crack_sxy.npy")

# --- Superpose totals (Pa)
Sxx_tot = Sxx_bem + Sxx_cr
Syy_tot = Syy_bem + Syy_cr
Sxy_tot = Sxy_bem + Sxy_cr

# Save totals (Pa)
np.save(out_dir / "Sxx_tot.npy", Sxx_tot)
np.save(out_dir / "Syy_tot.npy", Syy_tot)
np.save(out_dir / "Sxy_tot.npy", Sxy_tot)

# ============================================================
# Plot TOTAL using the SAME “plotter-style” controls
# ============================================================

Xg, Yg = np.meshgrid(xs, ys, indexing="xy")  # meters

def _apply_clip_percentiles(Z, clip_percentiles):
    # Use core_mod helper if present; else fallback
    fn = getattr(core_mod, "apply_clip_percentiles", None)
    if callable(fn):
        return fn(Z, clip_percentiles)
    if clip_percentiles is None:
        return Z
    lo, hi = clip_percentiles
    finite = Z[np.isfinite(Z)]
    if finite.size == 0:
        return Z
    v0 = np.percentile(finite, lo)
    v1 = np.percentile(finite, hi)
    return np.clip(Z, v0, v1)

def _robust_vmin_vmax(Z, opts, ref=0.0):
    # Use core_mod helper if present; else reasonable fallback
    fn = getattr(core_mod, "robust_vmin_vmax", None)
    if callable(fn):
        return fn(Z, opts, ref=ref)

    # fallback: symmetric robust scaling around 0 using percentiles
    finite = Z[np.isfinite(Z)]
    if finite.size == 0:
        return -1.0, 1.0
    p = 99.0
    a = float(np.percentile(np.abs(finite), p))
    if a <= 0:
        a = float(np.nanmax(np.abs(finite))) or 1.0
    vmax = float(getattr(opts, "vmax_factor", 1.0)) * max(a, abs(ref))
    return -vmax, vmax

def _plot_total_like_plotter(Z_pa, comp: str, title: str, fname: str):
    # Convert to MPa (plotter convention)
    Z_mpa = np.asarray(Z_pa, float) * 1e-6

    # Optional percentile clipping if present on opts
    clip_pct = getattr(opts, "clip_percentiles", None)
    Zp = _apply_clip_percentiles(Z_mpa, clip_pct)

    # Use ref=0 for totals (prevents constant-remote assumptions)
    vmin, vmax = _robust_vmin_vmax(Zp, opts, ref=0.0)

    n_bands = int(getattr(opts, "n_bands", 40))
    levels = np.linspace(vmin, vmax, n_bands + 1)

    fig, ax = plt.subplots(figsize=(6.2, 5.5))

    cf = ax.contourf(
        Xg * 1e3, Yg * 1e3, Zp,
        levels=levels,
        cmap=getattr(opts, "cmap", "jet"),
        extend=str(getattr(opts, "extend", "both")),
    )

    # contour lines (like your crack plot)
    ax.contour(
        Xg * 1e3, Yg * 1e3, Zp,
        levels=levels,
        colors="k",
        linewidths=0.5,
        alpha=0.55,
    )

    cbar = fig.colorbar(cf, ax=ax, label="Stress [MPa]")

    ax.set_title(comp)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.axis("equal")
    ax.grid(True, alpha=0.15)

    fig.savefig(out_dir / fname, dpi=getattr(opts, "dpi", 300), bbox_inches="tight")
    plt.show()
    print("Saved:", out_dir / fname)

_plot_total_like_plotter(Sxx_tot, "sxx", "TOTAL σ_xx = BEM + Crack (uncoupled)", "total_sxx.png")
_plot_total_like_plotter(Syy_tot, "syy", "TOTAL σ_yy = BEM + Crack (uncoupled)", "total_syy.png")
_plot_total_like_plotter(Sxy_tot, "sxy", "TOTAL σ_xy = BEM + Crack (uncoupled)", "total_sxy.png")

print("Saved totals + plots to:", out_dir)
